# Projet : Stage
### Objectif

Pour les attributs spatiaux et temporels, on voudrait faire la même chose. Pour cette partie, il y a 3 étapes à faire : 

1. Identifier les attributs spatiaux / temporels. 
2. Trouver le niveau d'hiérarchie de chaque attributs selon les hiérarchies spatiales / temporelles
3. Identifier le niveau d'hiérarchie le plus fin parmis tous les attributs spatiaux / temporels en tant que granularité minimum de dataset; identifier l'attribut au niveau d'hiérarchie le plus haut parmis tous les attributs en tant que scope de dataset et donner la liste de ses valeurs distinctes. 

L'output final qu'on demande est un dossier json de métadonnée de tous les datasets.

## 1. Hiérarchisation des données

### 1.1. Arrondissement/Cantons/Communes/Departements/Regions

In [16]:
import json

In [17]:
def fillChamp(dicChamps, dic_hierarchisee, rang) :

    for i in range(len(dicChamps['features'])) :
        champ = dicChamps['features'][i]['properties']['nom']
        if champ not in dic_hierarchisee[rang] :
            dic_hierarchisee[rang].append(champ.lower())
            dic_hierarchisee[rang].append(dicChamps['features'][i]['properties']['code'])
        
    return 

def fillDictionnaire() :
    fichiers = ['arrondissements', 'cantons', 'communes', 'departements', 'regions']
    dic_hierarchisee = {}

    for fichier in fichiers :
        dic_hierarchisee[fichier] = []
        mon_json = open(f"Education/levels/france-geojson/{fichier}-avec-outre-mer.geojson")
        data = json.load(mon_json)
        mon_json.close()

        fillChamp(data, dic_hierarchisee, fichier)

    return dic_hierarchisee

In [18]:
dic_hierarchisee = fillDictionnaire()

### 1.2 Quartiers

In [19]:
import csv

In [20]:

fichier = open("liste-correspondance-qp2024-qp2015.csv", "r", encoding="utf-8")
reader = csv.reader(fichier, delimiter=";")
listeQuartiers = list(reader)[1:]

In [21]:
dic_hierarchisee['quartiers'] = []
for i in range(len(listeQuartiers)) :
    quartier = listeQuartiers[i][1]
    if quartier not in dic_hierarchisee['quartiers'] and quartier != "" :
        dic_hierarchisee['quartiers'].append(quartier.lower())

dic_hierarchisee['QP'] = []
for i in range(len(listeQuartiers)) :
    quartier = listeQuartiers[i][3]
    if quartier not in dic_hierarchisee['QP'] and quartier != "" :
        dic_hierarchisee['QP'].append(quartier.lower())

del i, listeQuartiers, quartier, fichier, reader

In [22]:
for key in dic_hierarchisee:
    dic_hierarchisee[key].sort()

## 2. Identification des attributs spatiaux dans un fichier csv/xlsx

### 2.1 Algorithme de recherche

Mon objectif est de parcourir un tableau en vérifiant à chaque cellule si elle appartient à une donnée de mon dictionnaire jusqu'à trouver la plus petite et la plus grande granularité

In [23]:
# Tableau rangé par granularité des champs
champs = ['QP', 'quartiers', 'arrondissements', 'cantons', 'communes', 'departements', 'regions']
champs.reverse()
dic_hierarchisee = {champ: dic_hierarchisee[champ] for champ in champs if champ in dic_hierarchisee}

### 2.2 Fichiers csv & xlsx

In [24]:
import os
import pandas as pd

def getFiles(origine):
    fichiers = []
    for dossier in os.walk(origine):
        for fichier in dossier[2]:
            if fichier.endswith('.csv') or fichier.endswith('.xlsx'):
                fichiers.append(os.path.join(dossier[0], fichier))
    return fichiers

documents = getFiles("Education")

In [25]:
hierarchie_champs = {champ: i for i, champ in enumerate(dic_hierarchisee.keys())}

In [27]:
for fichier in documents :
    liste_attributs_spatiaux = {}
    if fichier.endswith('csv') :
        fic = open(fichier, "r")
        tab = csv.reader(fic, delimiter=";")
        headers = next(tab)

        if len(headers) <= 1 :
            tab = csv.reader(fic, delimiter=",")
            headers = headers[0].split(",")
        
        tab = list(tab)
        fic.close()

        for i in range(20) :
            for j in range (len(tab[i])) :
                if isinstance(tab[i][j], str) :
                    tab[i][j] = tab[i][j].lower()
                for attribut, donnees in dic_hierarchisee.items() :
                    if tab[i][j] in donnees and tab[i][j] != "":
                        if headers[j] not in liste_attributs_spatiaux and attribut not in liste_attributs_spatiaux.values() :
                            liste_attributs_spatiaux[headers[j]] = attribut
        
        if len(liste_attributs_spatiaux) == 0 :
            le_plus_bas = None
            le_plus_haut = None
        else :
    
            le_plus_bas = 'regions'
            le_plus_haut = 'QP'
            for attribut, champ in liste_attributs_spatiaux.items() :
                if hierarchie_champs[champ] > hierarchie_champs[le_plus_bas] :
                    le_plus_bas = champ
                if hierarchie_champs[champ] < hierarchie_champs[le_plus_haut] :
                    le_plus_haut = champ
    
        print(f"Fichier : {fichier}")
        print(f'Liste des attributs spatiaux : {liste_attributs_spatiaux}')
        print(f'Attribut le plus bas : {le_plus_bas}')
        print(f'Attribut le plus haut : {le_plus_haut}')

Fichier : Education/csv/annuaire-de-leducation.csv
Liste des attributs spatiaux : {'Code_postal': 'communes', 'Code_commune': 'cantons', 'Code_academie': 'regions', 'Code_region': 'departements', 'Libelle_academie': 'arrondissements'}
Attribut le plus bas : arrondissements
Attribut le plus haut : regions
Fichier : Education/csv/fr-esr-insersup.csv
Liste des attributs spatiaux : {'Région': 'regions', 'Académie': 'communes', 'Nombre de sortants': 'departements'}
Attribut le plus bas : communes
Attribut le plus haut : regions
Fichier : Education/csv/formations.csv
Liste des attributs spatiaux : {'"acaeta"': 'regions', '"discipli"': 'departements'}
Attribut le plus bas : departements
Attribut le plus haut : regions
Fichier : Education/csv/fr-en-baccalaureat-par-academie.csv
Liste des attributs spatiaux : {'Code académie': 'regions', "Nombre d'inscrits": 'departements'}
Attribut le plus bas : departements
Attribut le plus haut : regions
Fichier : Education/csv/lycees-donnees-generales.csv
L

In [31]:
for attribut, champ in liste_attributs_spatiaux.items() :
    le_plus_bas = None
    print(attribut, champ)

## 3. Création des classes utiles a la structure de métadonnée par fichier

In [ ]:
import uml_class

fic_spatialscope = uml_class.DS_Spatial_Scope()
